# WESAD — 04 · Normalization Sweep: does per-subject calibration close the LOSO gap?

*Project: When Wearable Stress Models Fail — Subject Generalization and Distribution Shift.*

Self-contained: this notebook **runs the sweep and analyses it** — no external scripts. It answers the
actionable question the earlier notebooks built toward:

> Notebook 03 found a large **subject-generalization gap** (random-CV ≫ LOSO). Notebook 05 showed **why** —
> between-subject offsets confound pooled relationships (Simpson's paradox). If that diagnosis is right,
> **per-subject normalization** — an unsupervised, label-free calibration a wearable can do on first wear —
> should *shrink the LOSO gap* and *remove the leakage* a flexible model exploits.

Sweep = split {random, loso} × normalization {none, subject_z} × model {logreg, hgb} × seed {0..9} = 80 fits.
Small enough to run inline in well under a minute; every number is reported with a confidence interval.

> Input: `outputs/tables/wrist_features_w60_s60.csv`. numpy / pandas / scikit-learn only.

## 1 · Setup & load features

In [ ]:
from pathlib import Path
import itertools
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.base import clone
from sklearn.model_selection import StratifiedKFold, LeaveOneGroupOut
from sklearn.metrics import f1_score

TAB = Path("outputs/tables"); FIG = Path("outputs/figures"); FIG.mkdir(parents=True, exist_ok=True)
feat = pd.read_csv(TAB/"wrist_features_w60_s60.csv")

try:
    ft = pd.read_csv(TAB/"feature_trust.csv"); TRUSTED = ft.loc[ft.trusted,"feature"].tolist()
except FileNotFoundError:
    TRUSTED = ["eda_mean","eda_std","eda_min","eda_max","eda_slope","eda_scr_count",
               "temp_mean","temp_std","temp_min","temp_max","temp_slope",
               "hr_mean","acc_std","acc_energy","acc_mad","acc_max"]
TRUSTED = [f for f in TRUSTED if f in feat.columns]
print(f"{len(feat)} windows | {len(TRUSTED)} trusted features | {feat.subject.nunique()} subjects")

## 2 · Run the sweep inline

`subject_z` z-scores each subject by **their own** feature mean/std (label-free — mimics a wearable
auto-calibrating to its wearer; no label leakage). LOSO reports per-subject F1 so we can bootstrap a CI.

In [ ]:
SPLITS=["random","loso"]; NORMS=["none","subject_z"]; MODELS=["logreg","hgb"]; SEEDS=list(range(10))

def subject_zscore(df, feats):
    out=df.copy()
    out[feats]=df.groupby("subject")[feats].transform(lambda c:(c-c.mean())/(c.std(ddof=0)+1e-9))
    return out

def make_model(name, seed):
    if name=="logreg":
        return make_pipeline(StandardScaler(), LogisticRegression(max_iter=5000, class_weight="balanced"))
    return HistGradientBoostingClassifier(random_state=seed, class_weight="balanced")

def evaluate(split, norm, model, seed):
    df = subject_zscore(feat, TRUSTED) if norm=="subject_z" else feat
    X,y,g = df[TRUSTED].values, df["condition"].values, df["subject"].values
    est = make_model(model, seed)
    if split=="random":
        skf=StratifiedKFold(5, shuffle=True, random_state=seed); yp=np.empty_like(y)
        for tr,te in skf.split(X,y):
            yp[te]=clone(est).fit(X[tr],y[tr]).predict(X[te])
        return f1_score(y,yp,average="macro"), None
    logo=LeaveOneGroupOut(); per={}
    for tr,te in logo.split(X,y,g):
        m=clone(est).fit(X[tr],y[tr]); per[str(g[te][0])]=f1_score(y[te],m.predict(X[te]),average="macro")
    return float(np.mean(list(per.values()))), per

records, per_subject = [], []
grid=list(itertools.product(SPLITS,NORMS,MODELS,SEEDS))
for i,(sp,nm,mo,sd) in enumerate(grid):
    f1, per = evaluate(sp,nm,mo,sd)
    records.append(dict(split=sp,norm=nm,model=mo,seed=sd,macro_f1=f1))
    if per:
        for subj,v in per.items():
            per_subject.append(dict(split=sp,norm=nm,model=mo,seed=sd,subject=subj,macro_f1=v))
    if (i+1)%10==0: print(f"  {i+1:2d}/{len(grid)} configs done")

res = pd.DataFrame(records); ps = pd.DataFrame(per_subject)
print("\nsweep complete:", dict(res.groupby(["split","norm","model"]).size().rename("n")))

## 3 · Headline — LOSO macro-F1, raw vs subject-normalized (bootstrap CI over subjects)

In [ ]:
def boot_ci(vals, n=5000, seed=0):
    rng=np.random.default_rng(seed); vals=np.asarray(vals)
    bs=[rng.choice(vals,len(vals),replace=True).mean() for _ in range(n)]
    return vals.mean(), np.percentile(bs,2.5), np.percentile(bs,97.5)

loso = ps[ps.split=="loso"]
rows=[]
for (nm,mo),g in loso.groupby(["norm","model"]):
    per_subj=g.groupby("subject")["macro_f1"].mean().values
    m,lo,hi=boot_ci(per_subj)
    rows.append(dict(model=mo,norm=nm,loso_f1=m,ci_lo=lo,ci_hi=hi))
loso_tab=pd.DataFrame(rows).sort_values(["model","norm"]).reset_index(drop=True)
loso_tab.to_csv(TAB/"sweep_loso_norm.csv", index=False)
print(loso_tab.to_string(index=False))
for mo in loso_tab.model.unique():
    a=loso_tab[(loso_tab.model==mo)&(loso_tab.norm=="none")]["loso_f1"].values[0]
    b=loso_tab[(loso_tab.model==mo)&(loso_tab.norm=="subject_z")]["loso_f1"].values[0]
    print(f"  {mo}: subject_z {b-a:+.3f} macro-F1 vs raw ({'closes gap' if b>a else 'no gain'})")

In [ ]:
fig,ax=plt.subplots(figsize=(7,4.5))
models=loso_tab.model.unique(); x=np.arange(len(models)); w=0.35
for i,nm in enumerate(["none","subject_z"]):
    sub=loso_tab[loso_tab.norm==nm].set_index("model").reindex(models)
    err=np.vstack([sub.loso_f1-sub.ci_lo, sub.ci_hi-sub.loso_f1])
    ax.bar(x+(i-0.5)*w, sub.loso_f1, w, yerr=err, capsize=4,
           label=("raw features" if nm=="none" else "subject-normalized"), color=["#C44E52","#4C72B0"][i])
ax.set_xticks(x); ax.set_xticklabels(models); ax.set_ylabel("LOSO macro-F1 (95% CI)")
ax.set_title("Does per-subject normalization improve cross-subject generalization?")
ax.legend(); plt.tight_layout(); plt.savefig(FIG/"sweep_normalization.png", dpi=140); plt.show()

## 4 · Leakage restated with uncertainty — random-CV vs LOSO
The nb 03 story with CIs: random-CV varies over seeds; LOSO is bootstrapped over subjects. The gap per model
is the subject-leakage inflation. Watch it shrink under `subject_z` — with the offset removed, there's no
subject identity left to leak.

In [ ]:
summary=[]
for (nm,mo),_ in res.groupby(["norm","model"]):
    r=res[(res.split=="random")&(res.norm==nm)&(res.model==mo)].groupby("seed")["macro_f1"].mean().values
    rm,rlo,rhi=boot_ci(r)
    l=loso_tab[(loso_tab.norm==nm)&(loso_tab.model==mo)].iloc[0]
    summary.append(dict(norm=nm,model=mo,random_f1=rm,random_ci=f"[{rlo:.3f},{rhi:.3f}]",
                        loso_f1=l.loso_f1,loso_ci=f"[{l.ci_lo:.3f},{l.ci_hi:.3f}]",leakage_gap=rm-l.loso_f1))
summ=pd.DataFrame(summary).sort_values(["model","norm"]).reset_index(drop=True)
summ.to_csv(TAB/"sweep_leakage_summary.csv", index=False)
print(summ.to_string(index=False))
print("\nleakage_gap = random-CV minus honest LOSO (larger = more subject leakage exploited)")

## 4.5 · Deployment-faithful calibration: rest-period only vs idealized

`subject_z` above is the *idealized* calibration — it uses all of a user's windows to compute their mean/std.
A real device only has an initial **rest period**. Here each held-out user is calibrated on their first `n_cal`
baseline windows only, those windows are **excluded from scoring** (they were used to estimate the stats), and
the model predicts forward. This is the honest deployment number; expect it at or slightly below idealized,
and to improve with a longer calibration period.

In [ ]:
from sklearn.model_selection import LeaveOneGroupOut

def baseline_calibrate(df, feats, n_cal=5, cal_condition=1):
    df = df.reset_index(drop=True); out = df.copy(); gsd = df[feats].std(ddof=0); cal_idx = []
    for _, g in df.groupby("subject"):
        cal = g[g.condition == cal_condition].sort_values("t_start").head(n_cal)
        if len(cal) < 2: cal = g.sort_values("t_start").head(n_cal)
        cal_idx += list(cal.index)
        mu = df.loc[cal.index, feats].mean(); sd = df.loc[cal.index, feats].std(ddof=0).where(lambda s: s > 1e-6, gsd)
        out.loc[g.index, feats] = (df.loc[g.index, feats] - mu) / (sd + 1e-9)
    return out, df.index.isin(cal_idx)

def loso_rest_calibrated(df, feats, n_cal, seed=0):
    df = df.reset_index(drop=True); dfn, mask = baseline_calibrate(df, feats, n_cal)
    X, y, g = dfn[feats].values, df.condition.values, df.subject.values
    est = make_model("logreg", seed); per = {}
    for tr, te in LeaveOneGroupOut().split(X, y, g):
        te = te[~mask[te]]
        if len(te) == 0: continue
        per[str(g[te][0])] = f1_score(y[te], clone(est).fit(X[tr], y[tr]).predict(X[te]), average="macro")
    return np.array(list(per.values()))

_, raw_per   = evaluate("loso", "none", "logreg", 0)
_, ideal_per = evaluate("loso", "subject_z", "logreg", 0)
comp = [("raw (no calibration)", np.array(list(raw_per.values()))),
        ("idealized (all windows)", np.array(list(ideal_per.values())))]
for n_cal in [3, 5, 10]:
    comp.append((f"rest-calibrated (n={n_cal})", loso_rest_calibrated(feat, TRUSTED, n_cal)))

rows = [dict(method=name, loso_f1=(c:=boot_ci(v))[0], ci_lo=c[1], ci_hi=c[2]) for name, v in comp]
comp_tab = pd.DataFrame(rows); comp_tab.to_csv(TAB/"calibration_comparison.csv", index=False)
print(comp_tab.to_string(index=False))

plt.figure(figsize=(7,4))
err = np.vstack([comp_tab.loso_f1-comp_tab.ci_lo, comp_tab.ci_hi-comp_tab.loso_f1])
plt.bar(comp_tab.method, comp_tab.loso_f1, yerr=err, capsize=4,
        color=["#C44E52","#4C72B0","#8DA0CB","#66C2A5","#2E7D32"])
plt.ylabel("LOSO macro-F1 (95% CI)"); plt.xticks(rotation=25, ha="right")
plt.title("Idealized vs deployment-faithful (rest-period) calibration")
plt.tight_layout(); plt.savefig(FIG/"calibration_comparison.png", dpi=140); plt.show()

## 5 · Findings & handoff

Fill from your run:
- **Per-subject normalization moves LOSO macro-F1 by `[__]` (logreg) / `[__]` (hgb)**, CIs `[__]`. The
  Simpson's-paradox diagnosis turned into an intervention.
- **Leakage gap** shrinks from `[__]` (raw) to `[__]` (subject_z) — removing the between-subject offset removes
  the thing the flexible model was memorising.
- **Deployment reading:** a wearable that calibrates to each new user should expect performance nearer the
  `subject_z` LOSO number. This assumes per-user calibration data — which a longitudinal product like Eli's
  naturally has (the same person, every day).

**This completes the analysis (notebooks 01→05).** Next: package everything into the repo structure —
`src/` modules, `configs/`, `README`, environment file, model card. Upload `outputs/tables/sweep_*.csv` and
`outputs/figures/sweep_normalization.png` with the rest of your outputs.

*(At this scale the sweep runs inline in seconds. If you later expand the grid — more features sets, window
sizes, or models — the same `evaluate()` loop can be split across SLURM tasks, but you don't need that here.)*